In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
file_path = "../data/raw/online_retail_II.xlsx"

df = pd.read_excel(file_path)

print("Raw dataset loaded!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Raw dataset loaded!
Rows: 525461
Columns: 8


In [3]:
df_clean = df.copy()

print("Working copy created!")

Working copy created!


In [4]:
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"],
    errors="coerce"
)

print(df_clean["InvoiceDate"].dtype)

datetime64[ns]


In [5]:
duplicates = df_clean.duplicated().sum()

print("Duplicate records:", duplicates)

Duplicate records: 6865


In [6]:
df_clean = df_clean.drop_duplicates()

print("Rows after removing duplicates:", len(df_clean))

Rows after removing duplicates: 518596


In [7]:
missing = df_clean.isnull().sum()

missing

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107833
Country             0
dtype: int64

In [8]:
df_clean = df_clean.dropna(subset=["Customer ID"])

print("Rows after removing missing Customer IDs:", len(df_clean))

Rows after removing missing Customer IDs: 410763


In [9]:
df_clean = df_clean.dropna(subset=["Description"])

print("Rows after removing missing descriptions:", len(df_clean))

Rows after removing missing descriptions: 410763


In [10]:
df_clean["Is_Cancelled"] = (
    df_clean["Invoice"]
    .astype(str)
    .str.startswith("C")
)

print(df_clean["Is_Cancelled"].value_counts())

Is_Cancelled
False    400947
True       9816
Name: count, dtype: int64


In [11]:
df_clean = df_clean[
    df_clean["Is_Cancelled"] == False
].copy()

print("Rows after removing cancelled transactions:", len(df_clean))

Rows after removing cancelled transactions: 400947


In [12]:
df_clean.drop(columns=["Is_Cancelled"], inplace=True)

In [13]:
df_clean = df_clean[
    df_clean["Quantity"] > 0
].copy()

print("Rows after quantity validation:", len(df_clean))

Rows after quantity validation: 400947


In [14]:
df_clean = df_clean[
    df_clean["Price"] > 0
].copy()

print("Rows after price validation:", len(df_clean))

Rows after price validation: 400916


In [15]:
df_clean["Revenue"] = (
    df_clean["Quantity"] * df_clean["Price"]
)

In [16]:
df_clean[[
    "Invoice",
    "StockCode",
    "Quantity",
    "Price",
    "Revenue"
]].head()

,Invoice,StockCode,Quantity,Price,Revenue
0,489434,85048,12,6.95,83.4
1,489434,79323P,12,6.75,81.0
2,489434,79323W,12,6.75,81.0
3,489434,22041,48,2.10,100.8
4,489434,21232,24,1.25,30.0


In [17]:
df_clean["Year"] = df_clean["InvoiceDate"].dt.year

In [18]:
df_clean["Month"] = df_clean["InvoiceDate"].dt.month

In [19]:
df_clean["Month_Name"] = (
    df_clean["InvoiceDate"].dt.month_name()
)

In [20]:
df_clean["Quarter"] = (
    "Q" + df_clean["InvoiceDate"].dt.quarter.astype(str)
)

In [21]:
df_clean["Day_of_Week"] = (
    df_clean["InvoiceDate"].dt.day_name()
)

In [22]:
df_clean["Hour"] = df_clean["InvoiceDate"].dt.hour

In [23]:
customer_summary = (
    df_clean
    .groupby("Customer ID")
    .agg(
        Total_Revenue=("Revenue", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Total_Orders=("Invoice", "nunique"),
        Average_Order_Value=("Revenue", "mean"),
        Unique_Products=("StockCode", "nunique")
    )
    .reset_index()
)

In [24]:
customer_summary.head()

,Customer ID,Total_Revenue,Total_Quantity,Total_Orders,Average_Order_Value,Unique_Products
0,12346.0,372.86,70,11,11.298788,26
1,12347.0,1323.32,828,2,18.638310,70
2,12348.0,222.16,373,1,11.108000,20
3,12349.0,2671.14,993,3,26.187647,90
4,12351.0,300.93,261,1,14.330000,21


In [25]:
analysis_date = df_clean["InvoiceDate"].max()

print("Analysis date:", analysis_date)

Analysis date: 2010-12-09 20:01:00


In [26]:
rfm = (
    df_clean
    .groupby("Customer ID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x: (analysis_date - x.max()).days
        ),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

In [27]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,164,11,372.86
1,12347.0,2,2,1323.32
2,12348.0,73,1,222.16
3,12349.0,42,3,2671.14
4,12351.0,10,1,300.93


In [28]:
output_path = "../data/processed/cleaned_transactions.csv"

df_clean.to_csv(
    output_path,
    index=False
)

print("Cleaned dataset saved successfully!")
print(output_path)

Cleaned dataset saved successfully!
../data/processed/cleaned_transactions.csv


In [29]:
customer_summary.to_csv(
    "../data/processed/customer_summary.csv",
    index=False
)

print("Customer summary saved successfully!")

Customer summary saved successfully!


In [30]:
rfm.to_csv(
    "../data/processed/customer_rfm.csv",
    index=False
)

print("RFM dataset saved successfully!")

RFM dataset saved successfully!
